In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    auc,
    classification_report
)

# Create results directory to save plots
os.makedirs("results", exist_ok=True)

# ==========================================
# PHASE A: DATA PREPROCESSING
# ==========================================
print("--- Phase A: Loading and Preprocessing Dataset ---")

# 1. Load dataset directly from scikit-learn
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = Malignant (Cancer), 1 = Benign (No Cancer)

print(f"Dataset Shape: {df.shape}")
print(f"Missing Values: {df.isnull().sum().sum()} total missing values found.\n")

# 2. Separate features (X) and target (y)
X = df.drop('target', axis=1)
y = df['target']

# 3. Train-Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Plot 1: Target Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='target', hue='target', data=df, palette='Set2', legend=False)
plt.title('Target Class Distribution (0 = Malignant, 1 = Benign)')
plt.xlabel('Diagnosis (0 = Malignant, 1 = Benign)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=300)
plt.close()

# ==========================================
# PHASE B: MODEL IMPLEMENTATION
# ==========================================
print("--- Phase B: Training Models ---")

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Support Vector Machine": SVC(probability=True, random_state=42)
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f"{name} trained successfully.")

# ==========================================
# PHASE C: EVALUATION & VISUALIZATION
# ==========================================
print("\n--- Phase C: Evaluating Models ---")

results_list = []

# Setup figure for Confusion Matrices
fig_cm, axes_cm = plt.subplots(1, 3, figsize=(15, 4))

# Setup figure for ROC Curves
plt.figure(figsize=(8, 6))

for idx, (name, model) in enumerate(trained_models.items()):
    # Predictions
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results_list.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4)
    })
    
    print(f"\nClassification Report for {name}:")
    print(classification_report(y_test, y_pred, target_names=data.target_names))
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_cm[idx],
                xticklabels=['Malignant', 'Benign'],
                yticklabels=['Malignant', 'Benign'])
    axes_cm[idx].set_title(f'{name}\nConfusion Matrix')
    axes_cm[idx].set_xlabel('Predicted')
    axes_cm[idx].set_ylabel('Actual')
    
    # Plot ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

# Save Combined Confusion Matrix Plot
fig_cm.tight_layout()
fig_cm.savefig('results/confusion_matrices.png', dpi=300)
plt.close(fig_cm)

# Finalize and Save ROC Curve Plot
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/roc_curves.png', dpi=300)
plt.close()

# Create Comparison DataFrame
results_df = pd.DataFrame(results_list)
print("\n=== Final Model Comparison ===")
print(results_df.to_string(index=False))

# Save Metrics Comparison Bar Chart
results_melted = results_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(10, 6))
sns.barplot(x="Model", y="Score", hue="Metric", data=results_melted, palette="viridis")
plt.title("Model Performance Comparison Across Metrics")
plt.ylim(0.85, 1.02)
plt.ylabel("Score")
plt.legend(loc="lower right")
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=300)
plt.close()

print("\nAll plots have been successfully saved to the 'results/' folder!")

--- Phase A: Loading and Preprocessing Dataset ---
Dataset Shape: (569, 31)
Missing Values: 0 total missing values found.

--- Phase B: Training Models ---
Logistic Regression trained successfully.
Random Forest trained successfully.
Support Vector Machine trained successfully.

--- Phase C: Evaluating Models ---

Classification Report for Logistic Regression:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



C:\Users\USER\AppData\Local\Python\pythoncore-3.15-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Classification Report for Random Forest:
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114


Classification Report for Support Vector Machine:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


=== Final Model Comparison ===
                 Model  Accuracy  Precision  Recall  F1-Score
   Logistic Regression    0.9825     0.9861  0.9861    0.9861
         Random Forest    0.9561     0.9589  0.9722    0.9655
Support Vector Machine    0.9825   